# 13 — Profile Slices (interactive)

**Workflow:**
1. Run all setup cells (imports → load data → navigation).
2. Run the **Interactive Slicer** cell.
3. Select a mode button:
   - **Slicer** — click anywhere on the top-2 image to drop the orange slice bundle there.
   - **Max / Min** — click on the profile curve to place one or more peak markers.
   - **Undo** — removes the last Max or Min point placed.
4. Press **Save to JSON** when done.

Profile intensities are extracted from the **raw calibrated I/F** (CISSCAL VICAR output), not the CLEAR-subtracted residual.  
The top-2 image is still used for visual placement only.

> Requires `ipympl`: `pip install ipympl`

In [ ]:
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT   = Path('../')
STORAGE_ROOT   = Path('/mnt/storage')
PROJECT_STORAGE = STORAGE_ROOT / 'projects/Thesis/LuminousBandsInERing'

PIPELINE_DIR   = STORAGE_ROOT / 'pipeline_output'
CISSCAL_DIR    = STORAGE_ROOT / 'cisscal_output'
RAW_IMAGES_DIR = STORAGE_ROOT / 'raw_images'
OUT_DIR        = PROJECT_STORAGE / 'post_pipeline' / '13_profile_slices'

# ── Top-2 preprocessing (applied to residual for display only) ────────────────
HP_SIGMA     = 80.0
ARCSINH_MULT = 0.05

# ── Slice geometry ────────────────────────────────────────────────────────────
ANCHOR_MODE          = 'enceladus'     # 'enceladus', 'center', or 'max'
ANGLE_FIELD          = 'band_orientation'  # 'peak_theta', 'band_orientation'
FALLBACK_ANGLE_FIELD = 'peak_theta'
SOURCE_ANGLE_KIND    = 'auto'

# ── Profile extraction ────────────────────────────────────────────────────────
SLICE_SAMPLES = 2000
N_SLICES      = 11
SLICE_SPACING = 5.0
SMOOTH_WINDOW = 21

# ── Masks ─────────────────────────────────────────────────────────────────────
MASK_ENCELADUS = True
MASK_SATURN    = True

In [ ]:
from __future__ import annotations

import json
import math
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from scipy.ndimage import gaussian_filter, map_coordinates

## Configuration

In [ ]:
# ── Image to inspect ─────────────────────────────────────────────────────────
OPUSID = 'co-iss-n1608970715'

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT   = Path('../')
STORAGE_ROOT   = Path('/mnt/storage')
PROJECT_STORAGE = STORAGE_ROOT / 'projects/Thesis/LuminousBandsInERing'

PIPELINE_DIR   = STORAGE_ROOT / 'pipeline_output'
CISSCAL_DIR    = STORAGE_ROOT / 'cisscal_output'
RAW_IMAGES_DIR = STORAGE_ROOT / 'raw_images'
OUT_DIR        = PROJECT_STORAGE / 'post_pipeline' / '13_profile_slices'

# ── Top-2 preprocessing (applied to residual for display only) ────────────────
HP_SIGMA     = 80.0
ARCSINH_MULT = 0.05

# ── Slice geometry ────────────────────────────────────────────────────────────
ANGLE_FIELD          = 'null_peak_theta_64_deg'
FALLBACK_ANGLE_FIELD = 'main_peak_theta_deg'
SOURCE_ANGLE_KIND    = 'auto'

SLICE_SAMPLES = 2400
N_SLICES      = 15
SLICE_SPACING = 8.0
SMOOTH_WINDOW = 101

ANCHOR_MODE = 'enceladus'

MASK_ENCELADUS = True
MASK_SATURN    = True

## Helper functions

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# I/O
# ─────────────────────────────────────────────────────────────────────────────

def load_residual_npz(npz_path: Path) -> tuple[np.ndarray, np.ndarray]:
    arr  = np.load(npz_path)
    raw  = np.asarray(arr['residual'], dtype=np.float32)
    mask = np.asarray(arr['mask'], dtype=bool) if 'mask' in arr else np.isfinite(raw)
    raw  = np.nan_to_num(raw, nan=0.0)
    return raw, mask


# ─────────────────────────────────────────────────────────────────────────────
# Display images
# ─────────────────────────────────────────────────────────────────────────────

def build_top2(raw: np.ndarray, valid_mask: np.ndarray) -> np.ndarray:
    """HP-filtered + row/col median detrend + arcsinh stretch. Left panel."""
    hp   = (raw.astype(np.float64) - gaussian_filter(raw.astype(np.float64), sigma=HP_SIGMA)).astype(np.float32)
    work = hp.copy().astype(np.float64)
    work[~valid_mask] = np.nan
    with np.errstate(all='ignore'):
        rm = np.nanmedian(work, axis=1, keepdims=True)
        work -= np.where(np.isfinite(rm), rm, 0.0)
        cm = np.nanmedian(work, axis=0, keepdims=True)
        work -= np.where(np.isfinite(cm), cm, 0.0)
    work = np.nan_to_num(work, nan=0.0).astype(np.float32)
    work *= valid_mask.astype(np.float32)
    scale = max(float(np.std(work)) * ARCSINH_MULT, 1e-12)
    proc  = np.arcsinh(work / scale).astype(np.float32)
    fin   = np.isfinite(proc)
    lo, hi = np.percentile(proc[fin], [0.1, 99.9]) if fin.any() else (-1.0, 1.0)
    img = np.clip(proc, lo, hi)
    img = (img - img.min()) / (img.max() - img.min() + 1e-12)
    return img


# ─────────────────────────────────────────────────────────────────────────────
# Masks & geometry
# ─────────────────────────────────────────────────────────────────────────────

def build_analysis_mask(valid_mask, metrics, *, mask_enceladus, mask_saturn):
    h, w = valid_mask.shape
    m = valid_mask.astype(bool).copy()
    if metrics is None:
        return m
    if mask_enceladus:
        col, row, disc_r, fac = (
            metrics.get('disc_col'), metrics.get('disc_row'),
            metrics.get('disc_r_px'), metrics.get('cfg_mask_radius_factor', 1.5)
        )
        if all(isinstance(v, (int, float)) for v in (col, row, disc_r, fac)):
            rr = float(disc_r) * float(fac)
            yy, xx = np.mgrid[0:h, 0:w]
            m[(xx - float(col))**2 + (yy - float(row))**2 <= rr**2] = False
    if mask_saturn:
        c0, c1, r0, r1 = (
            metrics.get('saturn_mask_col_min'), metrics.get('saturn_mask_col_max'),
            metrics.get('saturn_mask_row_min'), metrics.get('saturn_mask_row_max')
        )
        if all(isinstance(v, (int, float)) for v in (c0, c1, r0, r1)):
            c0i = max(0, int(min(c0, c1))); c1i = min(w, int(max(c0, c1)))
            r0i = max(0, int(min(r0, r1))); r1i = min(h, int(max(r0, r1)))
            if c1i > c0i and r1i > r0i:
                m[r0i:r1i, c0i:c1i] = False
    return m


def choose_anchor(top2, mode, metrics=None):
    h, w = top2.shape
    if mode == 'enceladus' and metrics is not None:
        cx, cy = metrics.get('disc_col'), metrics.get('disc_row')
        if all(isinstance(v, (int, float)) for v in (cx, cy)):
            if 0.0 <= cx <= w - 1 and 0.0 <= cy <= h - 1:
                return float(cx), float(cy)
    if mode == 'center':
        return (w - 1) / 2.0, (h - 1) / 2.0
    iy, ix = np.unravel_index(int(np.argmax(top2)), top2.shape)
    return float(ix), float(iy)


def derive_slice_angle(theta_source, source_field, kind):
    if kind == 'auto':
        interpreted = 'peak_axis' if 'peak_theta' in (source_field or '').lower() else 'band_orientation'
    else:
        interpreted = kind
    theta_band  = float((theta_source + 90.0) % 180.0) if interpreted == 'peak_axis' else float(theta_source % 180.0)
    theta_slice = float((theta_band + 90.0) % 180.0)
    return theta_band, theta_slice, interpreted


# ─────────────────────────────────────────────────────────────────────────────
# Profile extraction from the CLEAR-subtracted residual
# ─────────────────────────────────────────────────────────────────────────────

def _sample_line(residual, mask, cx, cy, angle_deg, n_samples):
    h, w   = residual.shape
    a      = math.radians(float(angle_deg))
    dx, dy = math.cos(a), math.sin(a)
    L      = float(math.hypot(w, h))
    t      = np.linspace(-L, L, int(n_samples), dtype=np.float64)
    x = cx + t * dx
    y = cy + t * dy
    inb  = (x >= 0) & (x <= w - 1) & (y >= 0) & (y <= h - 1)
    vals = np.full_like(t, np.nan)
    if inb.any():
        idx    = np.where(inb)[0]
        raw_s  = map_coordinates(residual.astype(np.float64), [y[inb], x[inb]], order=1, mode='nearest')
        mask_s = map_coordinates(mask.astype(np.float32),     [y[inb], x[inb]], order=0, mode='nearest') > 0.5
        vals[idx[mask_s]] = raw_s[mask_s]
    good = np.isfinite(vals)
    return t, vals, x[good], y[good]


def extract_multi_slices(residual, mask, cx, cy, theta_slice_deg, n_samples, n_slices, spacing_px):
    n    = max(1, int(n_slices))
    offs = (np.arange(n) - (n - 1) / 2.0) * float(spacing_px)
    a    = math.radians(float(theta_slice_deg))
    band_px, band_py = -math.sin(a), math.cos(a)
    all_vals, lines, t_ref = [], [], None
    for o in offs:
        t, vals, xv, yv = _sample_line(residual, mask, cx + o*band_px, cy + o*band_py, theta_slice_deg, n_samples)
        if t_ref is None:
            t_ref = t
        all_vals.append(vals)
        lines.append((xv, yv))
    stack = np.vstack(all_vals)
    cnt   = np.sum(np.isfinite(stack), axis=0)
    mean  = np.divide(np.nansum(stack, axis=0), cnt,
                      out=np.full(cnt.shape, np.nan), where=cnt > 0)
    return (t_ref if t_ref is not None else np.array([])), mean, lines


def moving_average(y, window):
    w = max(1, int(window))
    if w % 2 == 0:
        w += 1
    k   = np.ones(w, dtype=np.float64)
    yv  = np.asarray(y, dtype=np.float64)
    fin = np.isfinite(yv)
    num = np.convolve(np.where(fin, yv, 0.0), k, mode='same')
    den = np.convolve(fin.astype(np.float64),  k, mode='same')
    return np.divide(num, den, out=np.full_like(num, np.nan), where=den > 0)


def _nearest_profile_point(t, ys, t_click):
    """Snap clicked t to nearest finite point on the smoothed profile."""
    valid    = np.isfinite(ys)
    t_v, y_v = t[valid], ys[valid]
    nearest  = int(np.argmin(np.abs(t_v - t_click)))
    return float(t_v[nearest]), float(y_v[nearest])

## Load data

In [ ]:
def _read_angle(metrics, primary, fallback):
    for key in (primary, fallback):
        v = metrics.get(key) if metrics else None
        if isinstance(v, (int, float)) and np.isfinite(v):
            return float(v), key
    return None, None


def _load_all(opusid: str):
    npz_path = CISSCAL_DIR  / opusid / 'residual.npz'
    mpath    = PIPELINE_DIR / opusid / 'metrics.json'

    residual, valid_mask = load_residual_npz(npz_path)
    metrics = json.loads(mpath.read_text()) if mpath.exists() else None

    src_angle, src_field = _read_angle(metrics, ANGLE_FIELD, FALLBACK_ANGLE_FIELD)
    if src_angle is None:
        raise RuntimeError(f'No angle found in metrics for {opusid}')

    theta_band, theta_slice, kind = derive_slice_angle(src_angle, src_field, SOURCE_ANGLE_KIND)
    analysis_mask = build_analysis_mask(valid_mask, metrics,
                                        mask_enceladus=MASK_ENCELADUS,
                                        mask_saturn=MASK_SATURN)
    top2     = build_top2(residual, analysis_mask)
    cx0, cy0 = choose_anchor(top2, ANCHOR_MODE, metrics)

    return dict(
        opusid=opusid,
        residual=residual,
        analysis_mask=analysis_mask,
        top2=top2,
        metrics=metrics,
        src_angle=src_angle, src_field=src_field,
        theta_band=theta_band, theta_slice=theta_slice, kind=kind,
        cx0=cx0, cy0=cy0,
    )


data = _load_all(OPUSID)
print(f"opusid      : {data['opusid']}")
print(f"src angle   : {data['src_angle']:.2f}° ({data['src_field']}, {data['kind']})")
print(f"theta_band  : {data['theta_band']:.2f}°")
print(f"theta_slice : {data['theta_slice']:.2f}°")
print(f"anchor      : ({data['cx0']:.1f}, {data['cy0']:.1f})")

## Navigate images

In [ ]:
# ── Load opusid list ──────────────────────────────────────────────────────────
_labels_path = DATA_ROOT / 'post_pipeline' / '11_bundle_A_review_labels.json'
_all_ids     = json.loads(_labels_path.read_text()).get('selected_opusids', [])
_n_total     = len(_all_ids)

def _has_picks(opusid: str) -> bool:
    return (OUT_DIR / opusid / 'picks.json').exists()

def _dropdown_label(opusid: str) -> str:
    return f"{'✓' if _has_picks(opusid) else '·'} {opusid}"

def _update_nav_status():
    idx  = _all_ids.index(_nav_dropdown.value) if _nav_dropdown.value in _all_ids else 0
    done = sum(1 for o in _all_ids if _has_picks(o))
    _nav_status.value = f'{idx + 1} / {_n_total}   ·   {done} done'

def _reload_data(opusid: str) -> None:
    global data
    try:
        data = _load_all(opusid)
    except Exception as exc:
        _nav_status.value = f'⚠  {exc}'
        return
    _nav_dropdown.options = [(_dropdown_label(o), o) for o in _all_ids]
    _nav_dropdown.value   = opusid
    _update_nav_status()
    try:
        _reset_viewer()
    except NameError:
        pass  # viewer cell not yet run


_nav_dropdown = widgets.Dropdown(
    options=[(_dropdown_label(o), o) for o in _all_ids],
    value=OPUSID,
    layout=widgets.Layout(width='400px'),
)
_prev_btn   = widgets.Button(description='◀ Prev', layout=widgets.Layout(width='90px'))
_next_btn   = widgets.Button(description='Next ▶', layout=widgets.Layout(width='90px'))
_load_btn   = widgets.Button(description='Load',   button_style='primary', layout=widgets.Layout(width='70px'))
_nav_status = widgets.Label('')

_nav_dropdown.observe(lambda c: _update_nav_status(), names='value')
_prev_btn.on_click(lambda b: _reload_data(_all_ids[max(0, _all_ids.index(_nav_dropdown.value) - 1)]))
_next_btn.on_click(lambda b: _reload_data(_all_ids[min(_n_total - 1, _all_ids.index(_nav_dropdown.value) + 1)]))
_load_btn.on_click(lambda b: _reload_data(_nav_dropdown.value))
_update_nav_status()

display(widgets.VBox([
    widgets.HBox([_prev_btn, _next_btn, _nav_dropdown, _load_btn, _nav_status]),
]))

In [ ]:
# ── Mutable state ──────────────────────────────────────────────────────────────
_state = {
    'cx': None, 'cy': None,
    't': None, 'raw_mean': None, 'ys': None,
    'lines': None,
    'max_picks': [],   # [{'x_px', 'y_px', 't_px'}, ...]
    'min_picks': [],
    'last_pick_type': None,
}


def _load_saved_picks(opusid: str) -> dict | None:
    """Return parsed picks.json for opusid, or None if it doesn't exist."""
    p = OUT_DIR / opusid / 'picks.json'
    return json.loads(p.read_text()) if p.exists() else None


def _reset_viewer():
    saved = _load_saved_picks(data['opusid'])
    if saved is not None:
        sl = saved.get('slicer', {})
        _state['cx']        = sl.get('cx_px', data['cx0'])
        _state['cy']        = sl.get('cy_px', data['cy0'])
        _state['max_picks'] = saved.get('picks', {}).get('max', [])
        _state['min_picks'] = saved.get('picks', {}).get('min', [])
        _status_lbl.value   = f"Loaded saved picks ({len(_state['max_picks'])} max, {len(_state['min_picks'])} min)"
    else:
        _state['cx']        = data['cx0']
        _state['cy']        = data['cy0']
        _state['max_picks'] = []
        _state['min_picks'] = []
        _status_lbl.value   = ''
    _state['last_pick_type'] = None
    _mode_toggle.value       = 'Slicer'
    _save_btn.description    = 'Save to JSON'
    _save_btn.button_style   = 'success'
    _update_profile()
    _redraw()


def _update_profile():
    cx, cy = _state['cx'], _state['cy']
    t, mean, lines = extract_multi_slices(
        data['residual'], data['analysis_mask'],
        cx, cy, data['theta_slice'], SLICE_SAMPLES, N_SLICES, SLICE_SPACING,
    )
    _state['t']        = t
    _state['raw_mean'] = mean
    _state['ys']       = moving_average(mean, SMOOTH_WINDOW)
    _state['lines']    = lines


def _t_to_img(t_val):
    """Back-project a profile t coordinate to image (x, y)."""
    a = math.radians(data['theta_slice'])
    return (
        _state['cx'] + t_val * math.cos(a),
        _state['cy'] + t_val * math.sin(a),
    )


# ── Figures ────────────────────────────────────────────────────────────────────
fig, (ax_left, ax_right) = plt.subplots(
    1, 2, figsize=(14, 6.5), dpi=100,
    gridspec_kw={'width_ratios': [1, 1.3]},
)
plt.tight_layout(pad=2.0)
display(fig)


def _draw_left():
    ax_left.clear()
    ax_left.imshow(data['top2'], cmap='gray', origin='upper')
    cx, cy = _state['cx'], _state['cy']
    lines  = _state['lines'] or []
    n      = len(lines)
    for i, (xv, yv) in enumerate(lines):
        if xv.size > 1:
            centre = (i == n // 2)
            ax_left.plot(xv, yv, color='#ff9800',
                         lw=2.2 if centre else 1.2,
                         alpha=0.95 if centre else 0.35)
    ax_left.scatter([cx], [cy], marker='+', s=140, c='#e91e63',
                    linewidths=2.0, zorder=6)
    for j, p in enumerate(_state['max_picks']):
        ax_left.scatter([p['x_px']], [p['y_px']], c='#d32f2f', s=60,
                        zorder=8, marker='o', linewidths=0)
        ax_left.annotate(f"max{j+1}", (p['x_px'], p['y_px']),
                         textcoords='offset points', xytext=(5, 5),
                         fontsize=7, color='#d32f2f')
    for j, p in enumerate(_state['min_picks']):
        ax_left.scatter([p['x_px']], [p['y_px']], c='#ff9800', s=60,
                        zorder=8, marker='o', linewidths=0)
        ax_left.annotate(f"min{j+1}", (p['x_px'], p['y_px']),
                         textcoords='offset points', xytext=(5, -10),
                         fontsize=7, color='#ff9800')
    ax_left.set_title(
        f"Top-2   θ_slice={data['theta_slice']:.1f}°   centre=({cx:.0f},{cy:.0f})",
        fontsize=9,
    )
    ax_left.axis('off')


def _draw_right():
    t, raw_mean, ys = _state['t'], _state['raw_mean'], _state['ys']
    ax_right.clear()
    if t is None or not np.isfinite(raw_mean).any():
        ax_right.set_title('No profile data', fontsize=9)
        return
    valid   = np.isfinite(raw_mean)
    t_valid = t[valid]
    xlim    = (float(t_valid[0]), float(t_valid[-1])) if t_valid.size > 1 else (t[0], t[-1])

    ax_right.plot(t, raw_mean, color='#90caf9', lw=1.0, alpha=0.6, label='Raw mean')
    ax_right.plot(t, ys,       color='#1565c0', lw=2.0,             label=f'Smoothed (w={SMOOTH_WINDOW})')
    ax_right.axhline(0, color='white', lw=0.7, ls='--', alpha=0.35)

    for j, p in enumerate(_state['max_picks']):
        lbl = f"max {j+1}" if len(_state['max_picks']) > 1 else 'max'
        ax_right.axvline(p['t_px'], color='#d32f2f', lw=1.0, ls='--', alpha=0.7)
        ax_right.scatter([p['t_px']], [ys[np.argmin(np.abs(t - p['t_px']))]],
                         c='#d32f2f', s=70, zorder=8, label=lbl)
    for j, p in enumerate(_state['min_picks']):
        lbl = f"min {j+1}" if len(_state['min_picks']) > 1 else 'min'
        ax_right.axvline(p['t_px'], color='#ff9800', lw=1.0, ls='--', alpha=0.7)
        ax_right.scatter([p['t_px']], [ys[np.argmin(np.abs(t - p['t_px']))]],
                         c='#ff9800', s=70, zorder=8, label=lbl)

    ax_right.set_xlim(*xlim)
    ax_right.set_xlabel('Distance along slice (px)', fontsize=9)
    ax_right.set_ylabel('CLEAR-subtracted residual', fontsize=9)
    ax_right.set_title(
        f"N={N_SLICES} slices, spacing={SLICE_SPACING} px   |   "
        f"Max: {len(_state['max_picks'])}   Min: {len(_state['min_picks'])}",
        fontsize=9,
    )
    ax_right.grid(True, alpha=0.22)
    if _state['max_picks'] or _state['min_picks']:
        ax_right.legend(fontsize=8, loc='best')


def _redraw():
    _draw_left()
    _draw_right()
    fig.canvas.draw_idle()


# ── Click handler ──────────────────────────────────────────────────────────────
def _on_click(event):
    if event.xdata is None or event.ydata is None:
        return
    mode = _mode_toggle.value

    if mode == 'Slicer' and event.inaxes == ax_left:
        _state['cx'] = event.xdata
        _state['cy'] = event.ydata
        _update_profile()
        _state['max_picks']      = []
        _state['min_picks']      = []
        _state['last_pick_type'] = None
        _redraw()
        _save_btn.description  = 'Save to JSON'
        _save_btn.button_style = 'success'
        _status_lbl.value = f"Slicer at ({event.xdata:.0f}, {event.ydata:.0f}) px"

    elif mode in ('Max', 'Min') and event.inaxes == ax_right:
        t_near, _ = _nearest_profile_point(_state['t'], _state['ys'], event.xdata)
        img_x, img_y = _t_to_img(t_near)
        key  = mode.lower()
        pick = {'x_px': img_x, 'y_px': img_y, 't_px': t_near}
        _state[f'{key}_picks'].append(pick)
        _state['last_pick_type'] = key
        _redraw()
        _save_btn.description  = 'Save to JSON'
        _save_btn.button_style = 'success'
        _status_lbl.value = (
            f"{mode} pick #{len(_state[f'{key}_picks'])}: "
            f"image=({img_x:.1f}, {img_y:.1f}) px   t={t_near:.1f} px"
        )


fig.canvas.mpl_connect('button_press_event', _on_click)


# ── Widgets ────────────────────────────────────────────────────────────────────
_mode_toggle = widgets.ToggleButtons(
    options=['Slicer', 'Max', 'Min'],
    value='Slicer',
    description='Mode:',
    button_style='',
    style={'button_width': '90px', 'description_width': 'initial'},
    tooltips=[
        'Click LEFT image to place the slice bundle',
        'Click RIGHT profile to mark a band peak',
        'Click RIGHT profile to mark a band trough',
    ],
)

_undo_btn = widgets.Button(
    description='Undo',
    button_style='warning',
    layout=widgets.Layout(width='80px'),
    tooltip='Remove the last Max or Min pick',
)

_save_btn = widgets.Button(
    description='Save to JSON',
    button_style='success',
    layout=widgets.Layout(width='140px'),
)

_save_plot_btn = widgets.Button(
    description='Save plot',
    button_style='info',
    layout=widgets.Layout(width='110px'),
)

_status_lbl = widgets.Label('')


def _on_undo(b):
    key = _state['last_pick_type']
    if not key or not _state[f'{key}_picks']:
        _status_lbl.value = 'Nothing to undo.'
        return
    removed = _state[f'{key}_picks'].pop()
    other   = 'min' if key == 'max' else 'max'
    if not _state[f'{key}_picks'] and _state[f'{other}_picks']:
        _state['last_pick_type'] = other
    elif not _state[f'{key}_picks']:
        _state['last_pick_type'] = None
    _redraw()
    _status_lbl.value = (
        f"Undid last {key} pick at ({removed['x_px']:.1f}, {removed['y_px']:.1f}) px"
    )


def _on_save_json(b):
    opusid   = data['opusid']
    out_path = OUT_DIR / opusid / 'picks.json'
    out_path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        'timestamp_utc': datetime.now(timezone.utc).isoformat(),
        'opusid': opusid,
        'slicer': {
            'cx_px':              _state['cx'],
            'cy_px':              _state['cy'],
            'theta_slice_deg':    data['theta_slice'],
            'theta_band_deg':     data['theta_band'],
            'theta_source_deg':   data['src_angle'],
            'theta_source_field': data['src_field'],
            'n_slices':           N_SLICES,
            'slice_spacing_px':   SLICE_SPACING,
        },
        'picks': {
            'max': _state['max_picks'],
            'min': _state['min_picks'],
        },
    }
    out_path.write_text(json.dumps(payload, indent=2))
    _save_btn.description  = '✓ Saved'
    _save_btn.button_style = 'success'
    _status_lbl.value      = f'Saved → {out_path}'
    try:
        _nav_dropdown.options = [(_dropdown_label(o), o) for o in _all_ids]
        _nav_dropdown.value   = opusid
        _update_nav_status()
    except NameError:
        pass


def _on_save_plot(b):
    plot_path = OUT_DIR / data['opusid'] / 'picks_overview.png'
    plot_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(plot_path, dpi=150, bbox_inches='tight')
    _save_plot_btn.description = '✓ Saved'
    _status_lbl.value = f'Plot saved → {plot_path}'


_undo_btn.on_click(_on_undo)
_save_btn.on_click(_on_save_json)
_save_plot_btn.on_click(_on_save_plot)

_reset_viewer()

display(widgets.VBox([
    widgets.HBox([_mode_toggle, _undo_btn]),
    widgets.HBox([_save_btn, _save_plot_btn, _status_lbl]),
]))


## Normalised-residual contrast (test)

Take the full CLEAR-subtracted residual (unmasked) and normalise it so that:
- **0** = global finite minimum of the image
- **1** = patch-mean at the best max pick coordinate

$$\tilde{r} = \frac{r - r_{\min}}{v_{\max} - r_{\min}}$$

By construction $I_{\max} = 1$. The contrast is then:

$$I_{\min} = \min\bigl(\text{all sampled min picks}\bigr), \qquad
\Delta = 1 - I_{\min}, \qquad
M = \frac{\Delta}{1 + I_{\min}}$$

This guarantees $I_{\max} = 1$ regardless of image-to-image residual magnitude, and $I_{\min}$ is always $\leq 1$ as long as the min picks are genuinely lower in the residual than the band peak — making negative contrasts physically impossible for well-placed picks.

In [ ]:
_picks     = json.loads((OUT_DIR / data['opusid'] / 'picks.json').read_text())
_max_picks = _picks['picks']['max']
_min_picks = _picks['picks']['min']
_slicer    = _picks['slicer']

# Re-extract the same smoothed profile the user saw when placing picks
_t, _raw_mean, _ = extract_multi_slices(
    data['residual'], data['analysis_mask'],
    _slicer['cx_px'], _slicer['cy_px'],
    _slicer['theta_slice_deg'],
    SLICE_SAMPLES,
    _slicer.get('n_slices', N_SLICES),
    _slicer.get('slice_spacing_px', SLICE_SPACING),
)
_ys = moving_average(_raw_mean, SMOOTH_WINDOW)

# Normalise profile to [0, 1] over its full finite range
_fin_p = np.isfinite(_ys)
_p_min = float(_ys[_fin_p].min())
_p_max = float(_ys[_fin_p].max())
_ys_n  = (_ys - _p_min) / (_p_max - _p_min)

def _pick_val(t_pick):
    valid = np.isfinite(_ys_n)
    t_v, y_v = _t[valid], _ys_n[valid]
    return float(y_v[np.argmin(np.abs(t_v - t_pick))])

_max_vals = np.array([_pick_val(p['t_px']) for p in _max_picks])
_min_vals = np.array([_pick_val(p['t_px']) for p in _min_picks])

# Best max pick
_best_idx = int(np.argmax(_max_vals))
_I_max    = float(_max_vals[_best_idx])
_t_max    = _max_picks[_best_idx]['t_px']

# Closest flanking min on each side; I_min = mean of the two
_left  = [(v, p) for v, p in zip(_min_vals, _min_picks) if p['t_px'] < _t_max]
_right = [(v, p) for v, p in zip(_min_vals, _min_picks) if p['t_px'] > _t_max]
_candidates = []
if _left:
    _candidates.append(min(_left,  key=lambda x: abs(x[1]['t_px'] - _t_max)))
if _right:
    _candidates.append(min(_right, key=lambda x: abs(x[1]['t_px'] - _t_max)))

_I_min     = float(np.mean([v for v, _ in _candidates]))
_best_mins = [p for v, p in _candidates]
_delta     = _I_max - _I_min

print(f"I_max = {_I_max:.4f}  (pick {_best_idx+1} of {len(_max_picks)})")
print(f"I_min = {_I_min:.4f}  (mean of {len(_candidates)} flanking picks)")
print(f"Δ     = {_delta:.4f}")

# ── Profile plot ───────────────────────────────────────────────────────────────
fig_r, ax_r = plt.subplots(figsize=(10, 4), dpi=100)
ax_r.plot(_t, _ys_n, color='#1565c0', lw=1.8)
ax_r.axhline(0, color='white', lw=0.7, ls='--', alpha=0.35)

for j, (p, v) in enumerate(zip(_max_picks, _max_vals)):
    chosen = (j == _best_idx)
    ax_r.scatter([p['t_px']], [v], c='#d32f2f', s=90 if chosen else 40, zorder=8,
                 linewidths=1.5 if chosen else 0,
                 edgecolors='white' if chosen else 'none')
    ax_r.annotate(f"max{j+1}={v:.3f}", (p['t_px'], v),
                  textcoords='offset points', xytext=(4, 6), fontsize=7, color='#d32f2f')

for j, (p, v) in enumerate(zip(_min_picks, _min_vals)):
    chosen = p in _best_mins
    ax_r.scatter([p['t_px']], [v], c='#ff9800', s=90 if chosen else 40, zorder=8,
                 linewidths=1.5 if chosen else 0,
                 edgecolors='white' if chosen else 'none')
    ax_r.annotate(f"min{j+1}={v:.3f}", (p['t_px'], v),
                  textcoords='offset points', xytext=(4, -14), fontsize=7, color='#ff9800')

ax_r.set_xlabel('Distance along slice (px)', fontsize=9)
ax_r.set_ylabel('Normalised profile [0, 1]', fontsize=9)
ax_r.set_title(
    f"{data['opusid']}\nI_max={_I_max:.4f}  I_min={_I_min:.4f}  Δ={_delta:.4f}",
    fontsize=9,
)
ax_r.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()
